# ACDC Pathology Classification

This notebook evaluates patient-level pathology classification on the ACDC
cardiac MRI dataset using two evaluation modes:

**Task**: Classify each patient into one of 5 cardiac pathologies:
NOR (Normal), DCM (Dilated Cardiomyopathy), HCM (Hypertrophic Cardiomyopathy),
MINF (Myocardial Infarction), RV (Right Ventricular Cardiomyopathy).

## Evaluation modes (set `EVAL_MODE` in the config cell)

### `"logreg"` — Logistic Regression Linear Probe
1. Extract one embedding per 2D slice from a **frozen** backbone
2. **Pool per patient**: mean-pool ED slices, mean-pool ES slices, concatenate → `(2 × embed_dim,)` feature
3. **Feature normalisation**: StandardScaler (zero mean, unit variance) fitted per CV fold
4. **C-sweep**: 10-fold stratified CV over 45 C values, pick best by mean CV accuracy
5. **Retrain** final pipeline (StandardScaler + LogisticRegression) on all training data, evaluate on held-out test set

### `"finetune"` — Fine-Tuning with Linear Head
1. Train a linear classification head on top of the backbone (optionally unfreezing the backbone)
2. Same patient-level pooling: mean-pool ED + mean-pool ES → concatenate → linear head
3. **LR sweep**: 10-fold stratified CV over 5 learning rates (AdamW + cosine annealing)
4. **Retrain** on all training data with best LR, evaluate on held-out test set
5. Set `FREEZE_BACKBONE = True` to train only the head, or `False` to fine-tune everything

**Supported backbones** (set `BACKBONE` in the config cell):
- `"cinema"` — Pretrained CineMA (`mathpluscode/CineMA`). Processes the 3D SAX volume, then
  decomposes the spatial token grid into per-slice groups and mean-pools each → one embedding per slice.
- `"dinov3"` — DINOv3 ViT (e.g. `dinov3_vits16`). Extracts the CLS token from each 2D slice independently.
- `"sam"` — SAM ViT (`facebook/sam-vit-base`). Runs the image encoder on each 2D slice and
  global-average-pools the spatial feature map → one embedding per slice.

Both modes also report **binary disease detection** (NOR vs disease) derived from the
5-way classifier output.

## 1. Imports and Configuration

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from cinema.segmentation.dataset import EndDiastoleEndSystoleDataset
from monai.transforms import ScaleIntensityd

from heartfm_evals.classification_probe import (
    C_POWER_RANGE,
    NUM_PATHOLOGIES,
    PATHOLOGY_CLASSES,
    PATHOLOGY_NAMES,
    PATHOLOGY_NAMES_LONG,
    build_patient_features,
    cache_cinema_cls_features,
    cache_cls_features,
    cache_sam_cls_features,
    evaluate_binary_detection,
    evaluate_classification,
    get_pathology_map,
    load_cached_cls_features,
    sweep_C_and_train,
)
from heartfm_evals.dense_linear_probe import MODEL_CONFIGS
from heartfm_evals.finetune_classification import (
    evaluate_finetune_classification,
    finetune_sweep_and_train,
)

/Users/fnanni/Projects/heartfm-evals/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ══════════════════════════════════════════════════════════
# CHANGE THIS TO SWITCH BACKBONE: "cinema", "dinov3", or "sam"
BACKBONE = "dinov3"
# CHANGE THIS TO SWITCH EVALUATION MODE: "logreg" or "finetune"
EVAL_MODE = "finetune"
# Only used when EVAL_MODE == "finetune":
# True = train only the linear head (frozen backbone)
# False = fine-tune backbone + head end-to-end
FREEZE_BACKBONE = False
# ══════════════════════════════════════════════════════════

assert EVAL_MODE in ("logreg", "finetune"), f"Unknown EVAL_MODE: {EVAL_MODE}"

# ── Paths ──
ACDC_DATA_DIR = Path("../data/heartfm/processed/acdc")

# ── DINOv3 settings (only used when BACKBONE == "dinov3") ──
DINOV3_REPO_DIR = "../models/dinov3/"
DINOV3_MODEL_NAME = "dinov3_vits16"  # swap to dinov3_vitb16/vitl16
DINOV3_WEIGHTS_PATH = f"../model_weights/{DINOV3_MODEL_NAME}.pth"

# ── CineMA settings (only used when BACKBONE == "cinema") ──
HF_CACHE_DIR = Path("../model_weights/hf")
AUTO_DOWNLOAD = True

# ── SAM settings (only used when BACKBONE == "sam") ──
SAM_MODEL_ID = "facebook/sam-vit-base"

# ── Derived names ──
if BACKBONE == "cinema":
    MODEL_NAME = "cinema_pretrained"
elif BACKBONE == "sam":
    MODEL_NAME = "sam_vit_base"
else:
    MODEL_NAME = DINOV3_MODEL_NAME
CLS_CACHE_DIR = Path(f"../cls_feature_cache/{MODEL_NAME}")

# ── Device ──
if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")

if BACKBONE in ("cinema", "sam"):
    HF_CACHE_DIR.mkdir(parents=True, exist_ok=True)

mode_desc = EVAL_MODE if EVAL_MODE == "logreg" else f"finetune (freeze_backbone={FREEZE_BACKBONE})"
print(f"Using device: {DEVICE}")
print(f"Backbone: {BACKBONE} ({MODEL_NAME})")
print(f"Eval mode: {mode_desc}")
print(f"CLS cache dir: {CLS_CACHE_DIR.resolve()}")

Using device: mps
Backbone: dinov3 (dinov3_vits16)
Eval mode: finetune (freeze_backbone=False)
CLS cache dir: /Users/fnanni/Projects/heartfm-evals/cls_feature_cache/dinov3_vits16


## 2. Load ACDC Metadata

In [3]:
train_meta_df = pd.read_csv(ACDC_DATA_DIR / "train_metadata.csv")
test_meta_df = pd.read_csv(ACDC_DATA_DIR / "test_metadata.csv")

print(f"Training set: {len(train_meta_df)} patients")
print(f"Test set:     {len(test_meta_df)} patients")
print(f"\nPathology distribution (train):\n{train_meta_df['pathology'].value_counts().to_string()}")
print(f"\nPathology distribution (test):\n{test_meta_df['pathology'].value_counts().to_string()}")

Training set: 100 patients
Test set:     50 patients

Pathology distribution (train):
pathology
DCM     20
HCM     20
MINF    20
NOR     20
RV      20

Pathology distribution (test):
pathology
DCM     10
NOR     10
MINF    10
HCM     10
RV      10


## 3. Create CineMA Datasets

In [4]:
transform = ScaleIntensityd(keys="sax_image", factor=1 / 255, channel_wise=False)

train_cinema = EndDiastoleEndSystoleDataset(
    data_dir=ACDC_DATA_DIR / "train",
    meta_df=train_meta_df,
    views="sax",
    transform=transform,
)

test_cinema = EndDiastoleEndSystoleDataset(
    data_dir=ACDC_DATA_DIR / "test",
    meta_df=test_meta_df,
    views="sax",
    transform=transform,
)

print(f"Train CineMA dataset: {len(train_cinema)} samples (ED+ES volumes)")
print(f"Test CineMA dataset:  {len(test_cinema)} samples")

Train CineMA dataset: 200 samples (ED+ES volumes)
Test CineMA dataset:  100 samples


## 4. Load Backbone

For `logreg` mode the backbone is frozen and used only for feature extraction.
For `finetune` mode the backbone may be unfrozen during training (controlled by `FREEZE_BACKBONE`).

In [5]:
if BACKBONE == "cinema":
    from cinema import CineMA

    backbone = CineMA.from_pretrained(
        cache_dir=str(HF_CACHE_DIR),
        local_files_only=not AUTO_DOWNLOAD,
    )
    EMBED_DIM = backbone.enc_down_dict["sax"].patch_embed.proj.out_features

elif BACKBONE == "sam":
    from transformers import SamImageProcessor, SamModel

    sam_image_processor = SamImageProcessor.from_pretrained(
        SAM_MODEL_ID,
        cache_dir=str(HF_CACHE_DIR),
        local_files_only=not AUTO_DOWNLOAD,
    )
    backbone = SamModel.from_pretrained(
        SAM_MODEL_ID,
        cache_dir=str(HF_CACHE_DIR),
        local_files_only=not AUTO_DOWNLOAD,
    )
    EMBED_DIM = backbone.config.vision_config.output_channels  # 256 for sam-vit-base

else:
    backbone = torch.hub.load(
        DINOV3_REPO_DIR, DINOV3_MODEL_NAME,
        source="local", weights=DINOV3_WEIGHTS_PATH,
    )
    EMBED_DIM = MODEL_CONFIGS[DINOV3_MODEL_NAME]["embed_dim"]

backbone.eval().to(DEVICE)

# Freeze backbone parameters (logreg always frozen; finetune respects FREEZE_BACKBONE)
if EVAL_MODE == "logreg" or FREEZE_BACKBONE:
    for p in backbone.parameters():
        p.requires_grad = False
    freeze_status = "frozen"
else:
    freeze_status = "trainable"

print(f"Loaded {MODEL_NAME} with {sum(p.numel() for p in backbone.parameters()):,} parameters ({freeze_status})")
print(f"embed_dim: {EMBED_DIM}")
print(f"Patient feature dim: {2 * EMBED_DIM} (ED-mean + ES-mean)")

Loaded dinov3_vits16 with 21,601,152 parameters (trainable)
embed_dim: 384
Patient feature dim: 768 (ED-mean + ES-mean)


## 5. Cache Per-Slice Features to Disk (logreg only)

Since the backbone is **frozen** in logreg mode, we pre-extract one embedding per 2D slice
and cache to disk. All backbones produce the same cache format: `{"cls_token": Tensor(embed_dim,)}`
per `.pt` file, one file per slice. Re-running this cell skips slices already cached.

*Skipped in finetune mode — features are computed on-the-fly during training.*

In [6]:
if EVAL_MODE == "logreg":
    if BACKBONE == "cinema":
        cache_fn = lambda model, ds, cd, device: cache_cinema_cls_features(model, ds, cd, device=device)
    elif BACKBONE == "sam":
        cache_fn = lambda model, ds, cd, device: cache_sam_cls_features(model, sam_image_processor, ds, cd, device=device)
    else:
        cache_fn = lambda model, ds, cd, device: cache_cls_features(model, ds, cd, device=device)

    print("Caching training features...")
    train_manifest = cache_fn(backbone, train_cinema, CLS_CACHE_DIR / "train", DEVICE)

    print("\nCaching test features...")
    test_manifest = cache_fn(backbone, test_cinema, CLS_CACHE_DIR / "test", DEVICE)

    print(f"\nCached: {len(train_manifest)} train, {len(test_manifest)} test slices")

    sample = torch.load(train_manifest[0]["path"], weights_only=True)
    print(f"Per-slice embedding shape: {sample['cls_token'].shape}")
else:
    print("Skipped (finetune mode computes features on-the-fly)")

Skipped (finetune mode computes features on-the-fly)


## 6. Build Patient-Level Features (logreg only)

Load cached tokens and build per-patient features:
ED embedding ⊕ ES embedding → `(2 × embed_dim,)` feature vector.

*Skipped in finetune mode.*

In [7]:
train_pathology_map = get_pathology_map(train_meta_df)
test_pathology_map  = get_pathology_map(test_meta_df)

if EVAL_MODE == "logreg":
    train_cls = load_cached_cls_features(train_manifest)
    test_cls  = load_cached_cls_features(test_manifest)

    print(f"Loaded features for {len(train_cls)} train, {len(test_cls)} test patients")

    train_features, train_labels, train_pids = build_patient_features(train_cls, train_pathology_map)
    test_features, test_labels, test_pids    = build_patient_features(test_cls, test_pathology_map)

    print(f"\nTrain features: {train_features.shape} (dtype={train_features.dtype})")
    print(f"Test features:  {test_features.shape}")

    expected_dim = 2 * EMBED_DIM
    assert train_features.shape[1] == expected_dim, f"Expected {expected_dim}, got {train_features.shape[1]}"
    print(f"\nFeature dim check passed: {expected_dim}")

    for split_name, labels in [("Train", train_labels), ("Test", test_labels)]:
        counts = {PATHOLOGY_NAMES[i]: (labels == i).sum().item() for i in range(NUM_PATHOLOGIES)}
        print(f"{split_name} label distribution: {counts}")
else:
    print("Skipped (finetune mode works directly on the CineMA datasets)")

Skipped (finetune mode works directly on the CineMA datasets)


In [8]:
# ── Diagnostic: check pre-extracted feature distribution ──
if EVAL_MODE == "finetune":
    from heartfm_evals.finetune_classification import (
        _group_samples_by_patient,
        _preextract_all_features,
    )

    _patients = _group_samples_by_patient(train_cinema, train_pathology_map)
    _feats = _preextract_all_features(
        backbone, train_cinema, _patients, DEVICE, BACKBONE,
    )
    print(f"Shape:            {_feats.shape}")
    print(f"Mean:             {_feats.mean():.4f}")
    print(f"Std:              {_feats.std():.4f}")
    print(f"Min / Max:        {_feats.min():.4f} / {_feats.max():.4f}")
    print(f"Per-dim std range: [{_feats.std(dim=0).min():.4f}, {_feats.std(dim=0).max():.4f}]")
    print(f"Any NaN: {_feats.isnan().any()},  Any Inf: {_feats.isinf().any()}")
    del _patients, _feats  # free memory

Pre-extracting features: 100%|██████████| 100/100 [00:40<00:00,  2.46it/s]

Shape:            torch.Size([100, 768])
Mean:             0.0006
Std:              0.3705
Min / Max:        -1.5312 / 1.4868
Per-dim std range: [0.0529, 0.3008]
Any NaN: False,  Any Inf: False


## 7. Model Training

### Logistic Regression (`logreg` mode)

The raw CLS token features from different dimensions can have very different scales.
`StandardScaler` transforms each feature dimension to have **zero mean** and **unit
variance**. The scaler is fitted **only on training data** within each CV fold.

10-fold stratified CV sweeps 45 C values, picks best by mean accuracy, retrains on all data.

### Fine-Tuning (`finetune` mode)

Trains a linear classification head (+ optionally the backbone) using AdamW with
cosine annealing. 10-fold stratified CV sweeps 5 learning rates
(`1e-5, 5e-5, 1e-4, 5e-4, 1e-3`), with early stopping (patience=10).
Weight decay is fixed at `1e-4`. Retrains on all data with the best LR.

In [ ]:
if EVAL_MODE == "logreg":
    best_C, final_model, sweep_results = sweep_C_and_train(
        train_features, train_labels,
        n_folds=10,
    )
    print(f"\nBest C = {best_C:.4g}")
    print(f"Best mean CV accuracy = {max(r['mean_cv_acc'] for r in sweep_results):.4f}")

else:
    image_proc = sam_image_processor if BACKBONE == "sam" else None
    best_lr, backbone, ft_head, sweep_results = finetune_sweep_and_train(
        backbone=backbone,
        cinema_dataset=train_cinema,
        pathology_map=train_pathology_map,
        embed_dim=EMBED_DIM,
        device=DEVICE,
        backbone_type=BACKBONE,
        freeze_backbone=FREEZE_BACKBONE,
        image_processor=image_proc,
        n_folds=10,
    )
    print(f"\nBest LR = {best_lr:.4g}")
    print(f"Best mean CV accuracy = {max(r['mean_cv_acc'] for r in sweep_results):.4f}")

### Hyperparameter Sweep Curve

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4), dpi=150)

mean_accs = [r["mean_cv_acc"] for r in sweep_results]
std_accs = [r["std_cv_acc"] for r in sweep_results]

if EVAL_MODE == "logreg":
    x_values = [r["C"] for r in sweep_results]
    ax.semilogx(x_values, mean_accs, "o-", markersize=3, linewidth=1)
    ax.fill_between(
        x_values,
        [m - s for m, s in zip(mean_accs, std_accs)],
        [m + s for m, s in zip(mean_accs, std_accs)],
        alpha=0.2,
    )
    ax.axvline(best_C, color="red", linestyle="--", alpha=0.7, label=f"Best C={best_C:.2g}")
    ax.set_xlabel("Regularisation strength C")
    ax.set_title(f"Logistic Regression C-Sweep — 10-fold Stratified CV ({MODEL_NAME})")
else:
    x_values = [r["lr"] for r in sweep_results]
    ax.semilogx(x_values, mean_accs, "o-", markersize=5, linewidth=1.5)
    ax.fill_between(
        x_values,
        [m - s for m, s in zip(mean_accs, std_accs)],
        [m + s for m, s in zip(mean_accs, std_accs)],
        alpha=0.2,
    )
    ax.axvline(best_lr, color="red", linestyle="--", alpha=0.7, label=f"Best LR={best_lr:.2g}")
    ax.set_xlabel("Learning Rate")
    ax.set_title(f"Fine-Tune LR Sweep — 10-fold Stratified CV ({MODEL_NAME})")

ax.set_ylabel("Mean CV Accuracy")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Test Set Evaluation (5-Way Classification)

In [ ]:
if EVAL_MODE == "logreg":
    test_metrics = evaluate_classification(final_model, test_features, test_labels)
    test_pids_eval = test_pids
    test_labels_eval = test_labels
else:
    image_proc = sam_image_processor if BACKBONE == "sam" else None
    test_metrics = evaluate_finetune_classification(
        backbone, ft_head, test_cinema, test_pathology_map,
        DEVICE, BACKBONE, image_processor=image_proc,
    )
    test_pids_eval = test_metrics["pids"]
    test_labels_eval = torch.tensor(test_metrics["labels"], dtype=torch.long)

print(f"Test Top-1 Accuracy:          {test_metrics['accuracy']:.4f}")
print(f"Test Macro F1:                {test_metrics['macro_f1']:.4f}")
print(f"Test Macro Sensitivity (TPR): {test_metrics['macro_sensitivity']:.4f}")
print(f"Test Macro Specificity (TNR): {test_metrics['macro_specificity']:.4f}")
print("\nPer-class metrics:")
print(f"  {'Class':>4s}  {'Accuracy':>8s}  {'Specificity':>11s} {'Sensitivity':>11s}")
print(f"  {'-'*4}  {'-'*8}  {'-'*11}  {'-'*11}")
for cls in test_metrics["per_class_accuracy"]:
    acc  = test_metrics["per_class_accuracy"][cls]
    spec = test_metrics["per_class_specificity"][cls]
    sens = test_metrics["per_class_sensitivity"][cls]
    print(f"  {cls:>4s}  {acc:>8.4f}  {spec:>11.4f}  {sens:>11.4f}")

### ROC AUC Scores

In [ ]:
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import label_binarize

y_true_bin = label_binarize(test_labels_eval.numpy(), classes=list(range(NUM_PATHOLOGIES)))
y_prob = test_metrics["probabilities"]

macro_auc     = roc_auc_score(y_true_bin, y_prob, multi_class="ovr", average="macro")
per_class_auc = roc_auc_score(y_true_bin, y_prob, multi_class="ovr", average=None)

print(f"Macro ROC AUC (one-vs-rest): {macro_auc:.4f}\n")
print("Per-class ROC AUC:")
for cls_name, auc_val in zip(PATHOLOGY_CLASSES.keys(), per_class_auc, strict=False):
    print(f"  {cls_name:>4s}: {auc_val:.4f}")

In [ ]:
from sklearn.metrics import auc as sk_auc
from sklearn.metrics import roc_curve as sk_roc_curve

fig, axes = plt.subplots(1, NUM_PATHOLOGIES, figsize=(4 * NUM_PATHOLOGIES, 4), dpi=150)

for i, (cls_name, cls_idx) in enumerate(PATHOLOGY_CLASSES.items()):
    fpr, tpr, _ = sk_roc_curve(y_true_bin[:, cls_idx], y_prob[:, cls_idx])
    roc_auc_val = sk_auc(fpr, tpr)

    axes[i].plot(fpr, tpr, linewidth=2, label=f"AUC = {roc_auc_val:.3f}")
    axes[i].plot([0, 1], [0, 1], "k--", alpha=0.3)
    axes[i].set_xlabel("False Positive Rate")
    if i == 0:
        axes[i].set_ylabel("True Positive Rate")
    axes[i].set_title(cls_name)
    axes[i].legend(loc="lower right")
    axes[i].set_aspect("equal")
    axes[i].grid(True, alpha=0.3)

fig.suptitle(f"Per-Class ROC Curves — {MODEL_NAME}", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(4, 4), dpi=150)

class_names = [PATHOLOGY_NAMES_LONG[cls] for cls in PATHOLOGY_CLASSES.keys()]

bars = ax.bar(class_names, per_class_auc, edgecolor="black", linewidth=0.5)
ax.axhline(macro_auc, color="gray", linestyle="--", linewidth=1, label=f"Macro AUC = {macro_auc:.3f}")

for bar, val in zip(bars, per_class_auc, strict=False):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            f"{val:.3f}", ha="center", va="bottom", fontsize=9)

ax.set_ylabel("ROC AUC")
ax.set_title(f"Per-Class ROC AUC — {MODEL_NAME}")
ax.set_ylim(0, 1.1)
ax.legend()
ax.grid(True, alpha=0.3, axis="y")
ax.xaxis.set_tick_params(rotation=45, labelsize=5)
plt.tight_layout()
plt.show()

### Confusion Matrix

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6), dpi=150)

class_names = list(PATHOLOGY_CLASSES.keys())
cm = test_metrics["confusion_matrix"]

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names,
    ax=ax,
)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title(f"Confusion Matrix — {MODEL_NAME} (Acc={test_metrics['accuracy']:.2%})")
plt.tight_layout()
plt.show()

### Classification Report

In [ ]:
report_df = pd.DataFrame(test_metrics["classification_report"]).T
display(report_df.round(3))

## 10. Per-Patient Results

In [ ]:
results_df = pd.DataFrame({
    "pid": test_pids_eval,
    "true_label": [PATHOLOGY_NAMES[int(l)] for l in test_labels_eval],
    "predicted": [PATHOLOGY_NAMES[p] for p in test_metrics["predictions"]],
})
results_df["correct"] = results_df["true_label"] == results_df["predicted"]

print(f"Correct: {results_df['correct'].sum()} / {len(results_df)}")
print("\nMisclassified patients:")
display(results_df[~results_df["correct"]])

## 11. Binary Disease Detection

Derives binary disease detection (NOR vs disease) from the 5-way classifier
output: `P(disease) = 1 - P(NOR)`. No retraining is needed.

In [ ]:
binary_metrics = evaluate_binary_detection(
    test_metrics["probabilities"],
    test_labels_eval,
)

print("Binary Disease Detection (NOR vs Disease)")
print("=" * 50)
print(f"Accuracy:    {binary_metrics['accuracy']:.4f}")
print(f"F1 Score:    {binary_metrics['f1']:.4f}")
print(f"Sensitivity: {binary_metrics['sensitivity']:.4f}")
print(f"Specificity: {binary_metrics['specificity']:.4f}")
print(f"ROC AUC:     {binary_metrics['roc_auc']:.4f}")

In [ ]:
from sklearn.metrics import auc, roc_curve

fig, ax = plt.subplots(figsize=(5, 5), dpi=150)

fpr, tpr, _ = roc_curve(binary_metrics["binary_labels"], binary_metrics["binary_probs"])
roc_auc_val = auc(fpr, tpr)

ax.plot(fpr, tpr, linewidth=2, label=f"AUC = {roc_auc_val:.3f}")
ax.plot([0, 1], [0, 1], "k--", alpha=0.3)
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title(f"Binary Disease Detection ROC — {MODEL_NAME}")
ax.legend(loc="lower right")
ax.set_aspect("equal")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 12. Summary

In [ ]:
print("=" * 60)
print("ACDC Pathology Classification — Summary")
print("=" * 60)
print(f"Backbone:          {BACKBONE} ({MODEL_NAME})")
feat_desc = {
    "cinema": "Per-slice mean-pooled spatial tokens",
    "dinov3": "Final-layer CLS token",
    "sam": "Global-average-pooled image encoder features",
}[BACKBONE]
print(f"Feature type:      {feat_desc}")
print(f"embed_dim:         {EMBED_DIM}")
print(f"Pooling:           ED-mean + ES-mean → ({2*EMBED_DIM},)")

if EVAL_MODE == "logreg":
    print("Eval mode:         Logistic Regression (frozen backbone)")
    print("Normalisation:     StandardScaler (zero mean, unit variance)")
    print("Classifier:        sklearn LogisticRegression (L-BFGS, L2)")
    print("Model selection:   10-fold stratified CV")
    print(f"Best C:            {best_C:.4g}")
    print(f"Train patients:    {len(train_meta_df)}")
else:
    ft_mode = "head only (frozen backbone)" if FREEZE_BACKBONE else "backbone + head"
    print(f"Eval mode:         Fine-tune ({ft_mode})")
    print("Optimizer:         AdamW (weight_decay=1e-4)")
    print("Scheduler:         CosineAnnealingLR")
    print("Model selection:   10-fold stratified CV")
    print(f"Best LR:           {best_lr:.4g}")
    print(f"Train patients:    {len(train_meta_df)}")

print(f"Test patients:     {len(test_pids_eval)}")
print("─" * 60)
print("5-Way Classification:")
print(f"  Test Accuracy:     {test_metrics['accuracy']:.4f}")
print(f"  Test Macro F1:     {test_metrics['macro_f1']:.4f}")
print(f"  Macro ROC AUC:     {macro_auc:.4f}")
print("Binary Disease Detection:")
print(f"  Accuracy:          {binary_metrics['accuracy']:.4f}")
print(f"  Sensitivity:       {binary_metrics['sensitivity']:.4f}")
print(f"  Specificity:       {binary_metrics['specificity']:.4f}")
print(f"  ROC AUC:           {binary_metrics['roc_auc']:.4f}")
print("=" * 60)